# M2 — Audit et préparation des données DiagOps

## Mission

DiagOps doit intégrer trois nouvelles sources sur les équipements, les événements et les interventions de maintenance. Votre objectif est de **comprendre les données, établir un diagnostic, préparer une version exploitable et décider si elle peut être transmise à M3**.

Ce notebook est une **trame d'investigation**, pas un pas à pas ni une correction. Vous pouvez modifier son organisation, déplacer du code dans des fonctions ou des scripts et ajouter les contrôles que vous jugez utiles. Chaque constat important doit être accompagné d'une interprétation et d'une décision.

### Principes à respecter

- ne jamais modifier les fichiers reçus ;
- mesurer avant de corriger ;
- nommer et justifier chaque règle ajoutée ;
- ne supprimer ou corriger aucune ligne silencieusement ;
- conserver dans une quarantaine les cas à écarter ou à examiner ;
- distinguer un constat, une hypothèse et une décision.

## 0. Environnement et point de départ

Les chemins ci-dessous recherchent la racine du dépôt S04 et utilisent son `data_pack/`. Ils fonctionnent depuis le starter de référence comme depuis `work/M2/`. La variable d'environnement `DIAGOPS_DATA_DIR` permet d'utiliser un autre emplacement sans modifier le notebook.

In [ ]:
from pathlib import Path
import hashlib
import os

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

def first_existing(candidates):
    for candidate in candidates:
        path = Path(candidate).resolve()
        if path.exists():
            return path
    return Path(candidates[0]).resolve()

roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPOSITORY_ROOT = next((root for root in roots if (root / 'data_pack' / 'MANIFEST.yaml').is_file()), None)
if os.environ.get('DIAGOPS_DATA_DIR'):
    DATA_DIR = Path(os.environ['DIAGOPS_DATA_DIR']).resolve()
elif REPOSITORY_ROOT:
    DATA_DIR = REPOSITORY_ROOT / 'data_pack' / '2026-S1'
else:
    raise FileNotFoundError('Racine du dépôt S04 introuvable')
if os.environ.get('DIAGOPS_REFERENCE_DIR'):
    REFERENCE_DIR = Path(os.environ['DIAGOPS_REFERENCE_DIR']).resolve()
elif REPOSITORY_ROOT:
    REFERENCE_DIR = REPOSITORY_ROOT / 'data_pack' / '2026-S1' / 'reference_runs' / 'm1_for_m2'
else:
    raise FileNotFoundError('Référence M1 vers M2 introuvable')

DATA_DIR, REFERENCE_DIR

In [ ]:
SOURCE_FILES = {
    'equipment': DATA_DIR / 'equipment' / 'equipment.csv',
    'events': DATA_DIR / 'events' / 'events.csv',
    'maintenance': DATA_DIR / 'maintenance' / 'maintenance_history.csv',
}

missing_files = [str(path) for path in SOURCE_FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Fichiers M2 introuvables : {missing_files}')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

raw = {name: pd.read_csv(path) for name, path in SOURCE_FILES.items()}
source_inventory = pd.DataFrame([
    {
        'source': name,
        'file': path.name,
        'rows': len(raw[name]),
        'columns': raw[name].shape[1],
        'sha256': sha256(path),
    }
    for name, path in SOURCE_FILES.items()
])
source_inventory

**Trace initiale à conserver :** les empreintes ci-dessus permettent de démontrer que l'audit porte sur les fichiers reçus. Recalculez-les à la fin si vous avez un doute sur leur intégrité.

## 1. Comprendre les sources et leurs relations

Avant de chercher des erreurs, expliquez ce que représente une ligne dans chaque table, identifiez les clés et vérifiez les relations attendues.

Relations annoncées :

```text
events.equipment_id -> equipment.equipment_id
maintenance_history.equipment_id -> equipment.equipment_id
maintenance_history.event_id -> events.event_id
```

### Questions à traiter

- Que représente une ligne dans chaque fichier ?
- Quelle colonne devrait identifier une ligne de manière unique ?
- Quelles relations sont obligatoires pour le cas DiagOps ?
- Les types lus automatiquement correspondent-ils au schéma fourni ?
- Quelles limites des sources sont déjà documentées dans `SCHEMA.md` et `DATA_CARD.md` ?

In [ ]:
# Point de départ : aperçu technique. Complétez par vos observations.
for name, frame in raw.items():
    print(f'\n--- {name}: {frame.shape[0]} lignes x {frame.shape[1]} colonnes ---')
    display(frame.head(3))
    display(frame.dtypes.rename('dtype').to_frame().T)

### Votre cartographie

Consignez ici l'unité de chaque table, sa clé attendue, ses relations et son utilité pour DiagOps.

In [ ]:
source_map = pd.DataFrame(columns=[
    'source', 'une_ligne_represente', 'cle_attendue',
    'relations_utiles', 'usage_diagops', 'limites_connues'
])
# TODO : compléter source_map.
source_map

In [ ]:
# TODO : mesurer les identifiants absents ou dupliqués et les références orphelines.
# Présentez des nombres, les lignes concernées et l'impact possible.


## 2. Établir un diagnostic de qualité

Construisez un registre des contrôles avant de transformer les données. Une règle utile possède un identifiant stable, une justification et une décision prévue en cas d'échec.

L'audit doit notamment couvrir : colonnes et types attendus, identifiants, catégories fermées, cohérences temporelles, relations entre tables, valeurs impossibles et valeurs manquantes significatives.

In [ ]:
rule_register = pd.DataFrame(columns=[
    'rule_id', 'source', 'column', 'description',
    'justification', 'severity', 'decision_if_failed'
])
# TODO : inscrire les règles retenues avant ou pendant leur implémentation.
rule_register

### Résultats des contrôles

Pour chaque règle, indiquez le nombre de lignes examinées, le nombre d'échecs et quelques identifiants représentatifs. Ne corrigez pas encore les données dans cette partie.

In [ ]:
check_results = pd.DataFrame(columns=[
    'rule_id', 'source', 'rows_checked', 'failures',
    'sample_row_identifiers', 'comment'
])

# TODO : implémenter vos contrôles et alimenter check_results.
# Vous pouvez placer les fonctions dans ce notebook ou dans starter/src/.
check_results

### Quarantaine

Une anomalie ne doit pas disparaître. La quarantaine conserve le fichier source, la ligne, la règle en échec, la valeur observée, la raison et la décision. Une même ligne peut apparaître plusieurs fois si elle enfreint plusieurs règles.

In [ ]:
QUARANTINE_COLUMNS = [
    'source_file', 'row_identifier', 'rule_id', 'column',
    'observed_value', 'reason', 'decision'
]
quarantine = pd.DataFrame(columns=QUARANTINE_COLUMNS)

# TODO : alimentez la quarantaine à partir des contrôles en échec.
quarantine

### Interprétation du diagnostic

Répondez ici aux questions 2 à 6 du brief : lisibilité des données, problèmes d'identifiants et de relations, règles métier enfreintes, corrections certaines et cas nécessitant un avis métier.

**Votre analyse :**  
_À compléter à partir des résultats mesurés._

## 3. Examiner les informations personnelles et la couverture

L'objectif n'est ni une expertise juridique complète ni une étude statistique avancée. Vous devez repérer les informations relatives à des personnes, justifier leur traitement et vérifier que les catégories utiles ne sont pas absentes ou trompeusement peu représentées.

In [ ]:
# TODO : rechercher dans work_order_note les emails, téléphones et noms éventuels.
# Distinguez détection automatique, vérification humaine et décision de traitement.


**Décision sur les informations relatives à des personnes :**  
_Qu'avez-vous recherché ? Qu'avez-vous trouvé ? Que conservez-vous, masquez-vous ou placez-vous en quarantaine, et pourquoi ?_

In [ ]:
# TODO : calculer des effectifs et proportions utiles au minimum par :
# - site ;
# - type d'équipement ;
# - criticité ;
# - sévérité des événements.
# Signalez les groupes dont le faible effectif limite l'interprétation.


### Éclairage fourni par les erreurs historiques de M1

La référence M1 est un historique déjà consulté. Elle sert à rechercher des concentrations d'erreurs, **pas à démontrer une causalité ni à promouvoir un modèle**. Vérifiez la qualité des jointures avant toute comparaison.

In [ ]:
M1_ERRORS_PATH = REFERENCE_DIR / 'analyses' / 'matrice_erreurs_m1.csv'
if not M1_ERRORS_PATH.is_file():
    raise FileNotFoundError(f'Référence M1 introuvable : {M1_ERRORS_PATH}')

m1_errors = pd.read_csv(M1_ERRORS_PATH)
print(f'Référence M1 : {len(m1_errors)} cas, {m1_errors.shape[1]} colonnes')
display(m1_errors.head(3))

In [ ]:
# TODO : joindre prudemment la référence M1 aux équipements et/ou événements.
# Comparez quelques comptages ou taux simples par catégorie.
# Documentez les lignes non appariées et évitez les conclusions sur de petits effectifs.


**Votre analyse de couverture et des erreurs M1 :**  
_Quelles catégories sont peu représentées ? Certaines erreurs semblent-elles concentrées ? Pourquoi ce constat reste-t-il limité ?_

## 4. Préparer une version exploitable

Travaillez sur des copies en mémoire. Pour chaque transformation, conservez l'état initial, la règle appliquée, le nombre de lignes concernées et la justification.

Une correction automatique est adaptée uniquement lorsqu'elle est certaine et reproductible. Les cas ambigus doivent rester visibles et être mis à l'écart ou signalés.

In [ ]:
processed = {name: frame.copy(deep=True) for name, frame in raw.items()}
transformation_log = pd.DataFrame(columns=[
    'source', 'rule_id', 'transformation', 'rows_affected',
    'before', 'after', 'justification'
])

# TODO : appliquer uniquement les transformations justifiées.
# N'écrivez les sorties qu'après avoir vérifié les résultats.
transformation_log

In [ ]:
# Contrôles de cohérence à compléter avant export.
preparation_summary = pd.DataFrame([
    {
        'source': name,
        'raw_rows': len(raw[name]),
        'prepared_rows': len(processed[name]),
        'quarantined_findings': 0,  # TODO : remplacer par votre mesure
        'status': 'a_verifier',
    }
    for name in raw
])
preparation_summary

### Export reproductible

Lorsque vos contrôles sont terminés, exportez les trois tables préparées et la quarantaine dans un dossier de sortie distinct. Le code d'export doit rester dans le notebook ou dans vos scripts afin qu'une autre personne puisse rejouer la préparation.

In [ ]:
# Exemple d'emplacement ; décommentez et adaptez seulement après validation.
# OUTPUT_DIR = Path('../output').resolve()
# PROCESSED_DIR = OUTPUT_DIR / 'processed'
# PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# processed['equipment'].to_csv(PROCESSED_DIR / 'equipment.csv', index=False)
# processed['events'].to_csv(PROCESSED_DIR / 'events.csv', index=False)
# processed['maintenance'].to_csv(PROCESSED_DIR / 'maintenance_history.csv', index=False)
# quarantine.to_csv(OUTPUT_DIR / 'quarantine.csv', index=False)


## 5. Diagnostic et décision

Répondez de manière synthétique aux dix questions du brief. Chaque réponse doit citer un résultat observable : nombre de lignes, taux, règle, tableau, graphique ou cas représentatif.

### 1. Sources et relations
_Que représente chaque fichier et comment les sources sont-elles reliées ?_

### 2. Présence et lisibilité
_Les données nécessaires sont-elles présentes et lisibles ?_

### 3. Identifiants, doublons et relations
_Quels problèmes avez-vous détectés et quel est leur impact ?_

### 4. Règles métier
_Quelles règles ne sont pas respectées ?_

### 5. Corrections certaines
_Qu'avez-vous pu corriger automatiquement et pourquoi ?_

### 6. Cas à examiner
_Qu'avez-vous mis en quarantaine ou laissé à l'expertise métier ?_

### 7. Informations relatives à des personnes
_Qu'avez-vous trouvé et quelle décision avez-vous prise ?_

### 8. Couverture
_Quelles catégories sont peu représentées ?_

### 9. Erreurs historiques de M1
_Certaines concentrations apparaissent-elles et quelles limites empêchent de conclure davantage ?_

### 10. Décision pour M3
_Les données sont-elles utilisables, utilisables sous conditions ou non utilisables en l'état ? Quelles conditions restent à satisfaire ?_

## Vérification finale

Avant de terminer, vérifiez que :

- les empreintes des sources reçues sont inchangées ;
- les contrôles peuvent être rejoués depuis un environnement propre ;
- des cas valides et invalides vérifient les règles importantes ;
- chaque anomalie écartée possède une raison compréhensible ;
- les trois tables préparées et la quarantaine sont produites ;
- les dix questions ont une réponse fondée sur des résultats ;
- la décision finale distingue faits, limites et conditions.

In [ ]:
final_source_checksums = {name: sha256(path) for name, path in SOURCE_FILES.items()}
initial_source_checksums = dict(zip(source_inventory['source'], source_inventory['sha256']))
assert final_source_checksums == initial_source_checksums, 'Un fichier source a été modifié pendant l audit'
print('Sources reçues inchangées : contrôle réussi.')